In [1]:
import os, json, httpx, asyncpg, asyncio
from dataclasses import dataclass, field
from enum import Enum
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv()

# ══════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════
FARMER_ID       = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
MANDI_JSON_PATH = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_URL   = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"
OPEN_METEO_URL  = "https://api.open-meteo.com/v1/forecast"
DB_URL          = os.getenv("DATABASE_URL")

# ── Testing caps ──────────────────────────────────────────
MAX_MESSAGES        = 10   # Trigger summary at this count
SUMMARIZE_AFTER     = 8    # Compress oldest 8 messages
KEEP_RECENT         = 2    # Always keep 2 latest messages

# ══════════════════════════════════════════════════════════
#  STATE MACHINE
# ══════════════════════════════════════════════════════════
class Stage(Enum):
    IDLE              = "idle"
    WAITING_FOR_CROP  = "waiting_for_crop"
    WAITING_FOR_MANDI = "waiting_for_mandi"

@dataclass
class ConvState:
    stage:             Stage  = Stage.IDLE
    pending_crop:      str    = ""
    available_mandis:  list   = field(default_factory=list)
    farmer_state:      str    = ""
    farmer_district:   str    = ""
    farmer_lat:        float  = 0.0
    farmer_lon:        float  = 0.0
    field_name:        str    = ""
    city:              str    = ""
    # Memory
    thread_id:         str    = ""   # = FARMER_ID (1:1 mapping)
    msg_count:         int    = 0
    summary:           str    = ""   # rolling summary from DB
    history:           list   = field(default_factory=list)  # [(role, text), ...]

# ══════════════════════════════════════════════════════════
#  LLM  (no tools bound — zero tool-call loops possible)
# ══════════════════════════════════════════════════════════
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    api_key=os.getenv("GROK_API_KEY"),
)

# ══════════════════════════════════════════════════════════
#  DB HELPERS
# ══════════════════════════════════════════════════════════
async def db_get_or_create_thread(farmer_id: str) -> str:
    """thread_id = farmer_id always → 1 farmer = 1 permanent thread."""
    conn = await asyncpg.connect(DB_URL)
    try:
        await conn.execute(
            """
            INSERT INTO chat_threads (thread_id, farmer_id, summary)
            VALUES ($1, $2::uuid, '')
            ON CONFLICT (thread_id) DO NOTHING
            """,
            farmer_id, farmer_id
        )
        row = await conn.fetchrow(
            "SELECT summary FROM chat_threads WHERE thread_id = $1", farmer_id
        )
        return row["summary"] or "" if row else ""
    finally:
        await conn.close()


async def db_save_message(thread_id: str, role: str, content: str):
    """Persist one message to chat_messages."""
    conn = await asyncpg.connect(DB_URL)
    try:
        await conn.execute(
            "INSERT INTO chat_messages (thread_id, role, content) VALUES ($1, $2, $3)",
            thread_id, role, content
        )
        await conn.execute(
            "UPDATE chat_threads SET updated_at = NOW() WHERE thread_id = $1",
            thread_id
        )
    finally:
        await conn.close()


async def db_load_recent_messages(thread_id: str, limit: int = 10) -> list:
    """Load recent messages from DB ordered ASC."""
    conn = await asyncpg.connect(DB_URL)
    try:
        rows = await conn.fetch(
            """
            SELECT role, content FROM chat_messages
            WHERE thread_id = $1
            ORDER BY created_at DESC LIMIT $2
            """,
            thread_id, limit
        )
        return [(r["role"], r["content"]) for r in reversed(rows)]
    finally:
        await conn.close()


async def db_save_summary(thread_id: str, summary: str):
    conn = await asyncpg.connect(DB_URL)
    try:
        await conn.execute(
            "UPDATE chat_threads SET summary = $2, updated_at = NOW() WHERE thread_id = $1",
            thread_id, summary
        )
    finally:
        await conn.close()

# ══════════════════════════════════════════════════════════
#  MEMORY — Summarise when msg_count >= MAX_MESSAGES
# ══════════════════════════════════════════════════════════
async def maybe_summarise(conv: ConvState):
    """Compress oldest 8 messages into a summary, keep 2 recent."""
    if conv.msg_count < MAX_MESSAGES:
        return

    print("\n📝 [Memory] Message limit reached — generating summary...")

    to_compress = conv.history[:SUMMARIZE_AFTER]
    to_keep     = conv.history[SUMMARIZE_AFTER:]

    convo_text = "\n".join(f"{r.upper()}: {c}" for r, c in to_compress)

    prompt = f"""You are summarising a KisanSaathi agricultural assistant conversation.

Previous summary (if any): {conv.summary or "None"}

New messages to add to summary:
{convo_text}

Write a concise summary (max 150 words) of what was discussed.
Include: crops mentioned, mandis checked, prices fetched, weather queries, decisions made.
Skip anything irrelevant. Do NOT say "This is a summary"."""

    resp = await llm.ainvoke([HumanMessage(content=prompt)])
    conv.summary = resp.content.strip()
    conv.history = to_keep
    conv.msg_count = len(to_keep)

    await db_save_summary(conv.thread_id, conv.summary)
    print(f"📝 [Memory] Summary saved ({len(conv.summary)} chars). Keeping {len(to_keep)} recent messages.")


def add_to_history(conv: ConvState, role: str, content: str):
    conv.history.append((role, content))
    conv.msg_count += 1


# ══════════════════════════════════════════════════════════
#  DATA FETCHERS  (Python calls APIs — LLM never does)
# ══════════════════════════════════════════════════════════
async def fetch_farmer_profile(farmer_id: str, conv: ConvState):
    """Load state/district/lat/lon from DB into conv state."""
    conn = await asyncpg.connect(DB_URL)
    try:
        farmer = await conn.fetchrow(
            "SELECT state_name, dist_name FROM farmers WHERE id = $1", farmer_id
        )
        field  = await conn.fetchrow(
            """SELECT field_name, city_name, center_lat, center_lon
               FROM farm_fields WHERE farmer_id = $1 LIMIT 1""",
            farmer_id
        )
    finally:
        await conn.close()

    if farmer:
        conv.farmer_state    = farmer["state_name"]
        conv.farmer_district = farmer["dist_name"]
    if field:
        conv.field_name  = field["field_name"]
        conv.city        = field["city_name"]
        conv.farmer_lat  = float(field["center_lat"])
        conv.farmer_lon  = float(field["center_lon"])


def get_mandis_for_location(state: str, district: str) -> list:
    with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    for s_key, districts in data.items():
        if s_key.lower() == state.lower():
            for d_key, mandis in districts.items():
                if d_key.lower() == district.lower():
                    return mandis
    return []


async def fetch_mandi_price(state: str, mandi_name: str, crop: str) -> str:
    """Call Agmarknet API. Always called fresh — never cached."""
    print(f"\n🌐 [API] Agmarknet → state={state} | mandi={mandi_name} | crop={crop}")
    clean = mandi_name.replace("APMC", "").strip()
    params = {
        "api-key":            os.getenv("AGMARKNET_API_KEY"),
        "format":             "json",
        "filters[state]":     state,
        "filters[market]":    clean,
        "filters[commodity]": crop,
    }
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
    except httpx.TimeoutException:
        return "⚠️  Government server ne respond nahi kiya. Thodi der baad try karein."

    if resp.status_code != 200:
        return f"⚠️  API Error {resp.status_code}."

    records = resp.json().get("records", [])
    if not records:
        return (
            f"⚠️  Aaj {crop} ka data {clean} mandi mein available nahi hai.\n"
            "Kal try karein ya koi aur mandi chunein."
        )

    r = records[0]
    return (
        f"✅ Aaj Ka Bhav\n"
        f"━━━━━━━━━━━━━━━━━━━━━━\n"
        f"🌾 Fasal  : {r['commodity']}\n"
        f"🏪 Mandi  : {r['market']}\n"
        f"💰 Bhav   : ₹{r['modal_price']} / Quintal\n"
        f"📅 Tarikh : {r['arrival_date']}\n"
        f"━━━━━━━━━━━━━━━━━━━━━━"
    )


async def fetch_weather_data(lat: float, lon: float, days: int) -> dict:
    """Call Open-Meteo. Always fresh — never cached."""
    print(f"\n🌐 [API] Open-Meteo → lat={lat} | lon={lon} | days={days}")
    params = {
        "latitude":      lat,
        "longitude":     lon,
        "daily":         "temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max",
        "forecast_days": days,
        "timezone":      "Asia/Kolkata",
    }
    async with httpx.AsyncClient() as client:
        resp = await client.get(OPEN_METEO_URL, params=params, timeout=15.0)
    resp.raise_for_status()
    return resp.json()


# ══════════════════════════════════════════════════════════
#  LLM HELPERS  (NLP only — read text, output text)
# ══════════════════════════════════════════════════════════
async def detect_intent(user_msg: str) -> dict:
    """Single LLM call: what type of query is this?"""
    prompt = f"""Analyze this farmer message and respond ONLY with JSON.

Message: "{user_msg}"

Rules:
- intent: one of ["price", "weather", "general"]
  • price   = asking about mandi price/bhav/rate/daam
  • weather = asking about mausam/barish/garmi/sardi/aandhi/temperature
  • general = anything else
- crop: crop name in English if mentioned, else ""
- days: number of days for weather (aaj=1, kal=2, teen din=3, hafte=7). 0 if not mentioned.

JSON only:
{{"intent": "price/weather/general", "crop": "", "days": 0}}"""

    resp = await llm.ainvoke([HumanMessage(content=prompt)])
    try:
        text = resp.content.strip().replace("```json","").replace("```","").strip()
        result = json.loads(text)
        days = int(result.get("days", 0))
        if days <= 0 or days > 16:
            days = 3
        return {
            "intent": result.get("intent", "general"),
            "crop":   result.get("crop", "").strip(),
            "days":   days,
        }
    except Exception:
        kw_price   = ["bhav","price","rate","daam","mandi","khareed","bech"]
        kw_weather = ["mausam","barish","baarish","weather","garmi","sardi","aandhi"]
        if any(k in user_msg.lower() for k in kw_price):
            return {"intent": "price",   "crop": "", "days": 3}
        if any(k in user_msg.lower() for k in kw_weather):
            return {"intent": "weather", "crop": "", "days": 3}
        return {"intent": "general", "crop": "", "days": 3}


async def extract_crop(user_msg: str) -> str:
    prompt = f"""Farmer was asked which crop they want. Their reply: "{user_msg}"
Extract ONLY the English crop name (e.g. Wheat, Onion, Tomato).
If no crop found respond: NONE"""
    resp = await llm.ainvoke([HumanMessage(content=prompt)])
    crop = resp.content.strip()
    return "" if crop.upper() == "NONE" else crop


def fuzzy_match_mandi(user_input: str, mandis: list) -> str:
    user_input = user_input.strip()
    if user_input.isdigit():
        idx = int(user_input) - 1
        return mandis[idx] if 0 <= idx < len(mandis) else ""
    for m in mandis:
        if user_input.lower() in m.lower() or m.lower() in user_input.lower():
            return m
    return ""


async def llm_answer_weather(user_msg: str, weather_data: dict,
                              field_name: str, city: str, days: int,
                              summary: str) -> str:
    """LLM analyses raw weather numbers and answers farmer's specific question."""
    daily      = weather_data.get("daily", {})
    dates      = daily.get("time", [])
    t_max      = daily.get("temperature_2m_max", [])
    t_min      = daily.get("temperature_2m_min", [])
    rain       = daily.get("precipitation_sum", [])
    wind       = daily.get("windspeed_10m_max", [])

    data_lines = []
    for i in range(min(days, len(dates))):
        data_lines.append(
            f"  {dates[i]}: Temp {t_min[i]}–{t_max[i]}°C | Rain {rain[i]}mm | Wind {wind[i]} km/h"
        )

    context = f"Previous context: {summary}" if summary else ""

    prompt = f"""Tu KisanSaathi hai — helpful agricultural assistant for Indian farmers.

{context}
Farmer ka khet: {field_name}, {city}
Farmer ka sawaal: "{user_msg}"

LIVE weather data ({days} din ke liye) — abhi API se fetch hua:
{chr(10).join(data_lines)}

Hinglish mein jawab de (Hindi + English mix):
1. Farmer ke sawaal ka SEEDHA jawab pehle
2. Numbers simple bhasha mein explain kar
3. Khet ke liye relevant advice de
4. 4-6 lines max, warm desi tone
Bullet points mat use karna."""

    resp = await llm.ainvoke([HumanMessage(content=prompt)])
    return resp.content.strip()


async def llm_general_answer(user_msg: str, summary: str, history: list) -> str:
    """LLM answers general farming questions using summary + recent history."""
    # Build recent context (last 4 exchanges at most)
    recent = history[-4:] if len(history) > 4 else history
    recent_text = "\n".join(f"{r.upper()}: {c}" for r, c in recent)

    context = ""
    if summary:
        context += f"Conversation summary:\n{summary}\n\n"
    if recent_text:
        context += f"Recent messages:\n{recent_text}"

    prompt = f"""Tu KisanSaathi hai — helpful agricultural assistant for Indian farmers.

{context}

Farmer ka naya sawaal: "{user_msg}"

Hinglish mein (Hindi + English) helpful answer de. Short rakh (max 4-5 lines).
Summary se context lo but CURRENT sawaal ka jawab priority hai."""

    resp = await llm.ainvoke([HumanMessage(content=prompt)])
    return resp.content.strip()


# ══════════════════════════════════════════════════════════
#  MAIN MESSAGE HANDLER  (the state machine)
# ══════════════════════════════════════════════════════════
async def handle_message(user_msg: str, conv: ConvState) -> str:

    # ── STAGE: Waiting for crop name ──────────────────────
    if conv.stage == Stage.WAITING_FOR_CROP:
        crop = await extract_crop(user_msg)
        if not crop:
            return "Maafi chahta hun, samajh nahi aaya. Kripya fasal ka naam batayein — jaise Wheat, Onion, Tomato."

        conv.pending_crop = crop
        mandis = get_mandis_for_location(conv.farmer_state, conv.farmer_district)

        if not mandis:
            conv.stage = Stage.IDLE
            return f"Aapke district {conv.farmer_district} ke liye mandi data abhi available nahi hai."

        conv.available_mandis = mandis
        conv.stage = Stage.WAITING_FOR_MANDI
        mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(mandis))
        return (
            f"📍 {conv.farmer_district} ({conv.farmer_state}) ki APMCs — {crop}:\n\n"
            f"{mandi_list}\n\n"
            "Kaunsi mandi ka bhav chahiye? (Number ya naam type karein)"
        )

    # ── STAGE: Waiting for mandi selection ────────────────
    if conv.stage == Stage.WAITING_FOR_MANDI:
        matched = fuzzy_match_mandi(user_msg, conv.available_mandis)
        if not matched:
            mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(conv.available_mandis))
            return f"Yeh mandi nahi mili. List mein se number ya naam chunein:\n\n{mandi_list}"

        # ✅ API called fresh — always live data
        result = await fetch_mandi_price(conv.farmer_state, matched, conv.pending_crop)

        conv.stage            = Stage.IDLE
        conv.pending_crop     = ""
        conv.available_mandis = []
        return result

    # ── STAGE: IDLE — detect what the farmer wants ─────────
    intent_data = await detect_intent(user_msg)
    intent = intent_data["intent"]

    # ── WEATHER QUERY ──────────────────────────────────────
    if intent == "weather":
        days = intent_data["days"]

        # Lazy-load farmer profile
        if not conv.farmer_lat:
            try:
                await fetch_farmer_profile(FARMER_ID, conv)
            except Exception as e:
                return f"⚠️ Database error: {e}"

        if not conv.farmer_lat:
            return "⚠️ Aapka khet registered nahi hai. Pehle khet register karein."

        try:
            # ✅ API called every time — always fresh, never stale
            weather_data = await fetch_weather_data(conv.farmer_lat, conv.farmer_lon, days)
        except Exception as e:
            return f"⚠️ Mausam server se data nahi aaya: {e}"

        return await llm_answer_weather(
            user_msg, weather_data, conv.field_name, conv.city, days, conv.summary
        )

    # ── MANDI PRICE QUERY ──────────────────────────────────
    if intent == "price":
        crop = intent_data["crop"]

        # Lazy-load farmer profile
        if not conv.farmer_state:
            try:
                await fetch_farmer_profile(FARMER_ID, conv)
            except Exception as e:
                return f"⚠️ Database error: {e}"

        if not crop:
            conv.stage = Stage.WAITING_FOR_CROP
            return "Aap kis fasal (crop) ka bhav jaanna chahte hain? (e.g. Wheat, Onion, Tomato)"

        # Crop known → show mandi list
        conv.pending_crop = crop
        mandis = get_mandis_for_location(conv.farmer_state, conv.farmer_district)

        if not mandis:
            return f"Aapke district {conv.farmer_district} ke liye mandi data abhi available nahi hai."

        conv.available_mandis = mandis
        conv.stage = Stage.WAITING_FOR_MANDI
        mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(mandis))
        return (
            f"📍 {conv.farmer_district} ({conv.farmer_state}) ki APMCs — {crop}:\n\n"
            f"{mandi_list}\n\n"
            "Kaunsi mandi ka bhav chahiye? (Number ya naam type karein)"
        )

    # ── GENERAL QUERY ──────────────────────────────────────
    # Summary used only for context — current question has priority
    return await llm_general_answer(user_msg, conv.summary, conv.history)


# ══════════════════════════════════════════════════════════
#  ENTRY POINT — Interactive chat with streaming simulation
# ══════════════════════════════════════════════════════════
async def chat_with_farmer():
    print(f"\n🌾 KisanSaathi | Farmer: {FARMER_ID}")
    print("━" * 50)

    # ── Load or create thread (1 farmer = 1 permanent thread)
    conv            = ConvState()
    conv.thread_id  = FARMER_ID   # thread_id IS farmer_id

    existing_summary = await db_get_or_create_thread(FARMER_ID)
    conv.summary     = existing_summary

    # ── Load recent messages from DB (resume chat)
    recent_msgs = await db_load_recent_messages(FARMER_ID, limit=KEEP_RECENT)
    conv.history  = recent_msgs
    conv.msg_count = len(recent_msgs)

    if conv.summary:
        print(f"📖 [Resuming chat] Summary loaded ({len(conv.summary)} chars)")
    if recent_msgs:
        print(f"📖 [Resuming chat] {len(recent_msgs)} recent messages loaded")
        print("\n--- Previous Messages ---")
        for role, content in recent_msgs:
            prefix = "👨‍🌾 Farmer" if role == "human" else "🤖 KisanSaathi"
            print(f"{prefix}: {content}")
        print("--- End of Previous Messages ---\n")

    print("\n🤖 KisanSaathi: Namaste! Main KisanSaathi hun. Aaj kya jaanna chahte hain?\n")

    while True:
        # ── Read farmer input (visible)
        try:
            user_input = input("👨‍🌾 Farmer: ").strip()
        except EOFError:
            break

        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "band karo", "bye"):
            print("🤖 KisanSaathi: Dhanyavad! Jai Jawan Jai Kisan 🌾")
            break

        # ── Process message
        print("🤖 KisanSaathi: ", end="", flush=True)  # streaming feel
        response = await handle_message(user_input, conv)

        # ── Streaming: print word by word
        for word in response.split(" "):
            print(word, end=" ", flush=True)
            await asyncio.sleep(0.03)
        print()   # newline after full response

        # ── Save to DB and in-memory history
        add_to_history(conv, "human", user_input)
        add_to_history(conv, "ai",    response)
        await db_save_message(conv.thread_id, "human", user_input)
        await db_save_message(conv.thread_id, "ai",    response)

        # ── Auto-summarise if message cap reached (testing: at 10)
        await maybe_summarise(conv)

        print()  # blank line between turns


# ── Run in Jupyter:
await chat_with_farmer()

# ── Run as .py:
# if __name__ == "__main__":
#     asyncio.run(chat_with_farmer())



🌾 KisanSaathi | Farmer: c59f6f44-1a98-4eaa-8cf0-3581316a32bb
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📖 [Resuming chat] Summary loaded (697 chars)
📖 [Resuming chat] 2 recent messages loaded

--- Previous Messages ---
👨‍🌾 Farmer: Accha mene kya kya Poocha??
🤖 KisanSaathi: Aapne poocha tha ki aapne kya kya poocha. Aapne pehle wheat ke bhav pooche the, phir onion ke bhav pooche, lekin Jarar mandi mein onion ka data available nahi tha.
--- End of Previous Messages ---


🤖 KisanSaathi: Namaste! Main KisanSaathi hun. Aaj kya jaanna chahte hain?

🤖 KisanSaathi: 📍 agra (Uttar Pradesh) ki APMCs — wheat:

  1. Achnera APMC
  2. Agra APMC
  3. Fatehabad APMC
  4. Fatehpur Sikri APMC
  5. Jagnair APMC
  6. Jarar APMC

Kaunsi mandi ka bhav chahiye? (Number ya naam type karein) 

🤖 KisanSaathi: 
🌐 [API] Agmarknet → state=Uttar Pradesh | mandi=Achnera APMC | crop=wheat
✅ Aaj Ka Bhav
━━━━━━━━━━━━━━━━━━━━━━
🌾 Fasal  : Wheat
🏪 Mandi  : Achnera APMC
💰 Bhav   : ₹2350 / Quintal
📅 Tarikh : 21/04/

### Stale State and other Things:

In [3]:
import os, json, httpx, asyncpg, asyncio
from dataclasses import dataclass, field
from datetime import datetime, timezone          # ← NEW
from enum import Enum
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

# ══════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════
FARMER_ID       = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
MANDI_JSON_PATH = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_URL   = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"
OPEN_METEO_URL  = "https://api.open-meteo.com/v1/forecast"
DB_URL          = os.getenv("DATABASE_URL")

MAX_MESSAGES    = 10
SUMMARIZE_AFTER = 8
KEEP_RECENT     = 2
GAP_MINUTES     = 10     # ← NEW: stale session threshold

# ══════════════════════════════════════════════════════════
#  STATE MACHINE
# ══════════════════════════════════════════════════════════
class Stage(Enum):
    IDLE              = "idle"
    WAITING_FOR_CROP  = "waiting_for_crop"
    WAITING_FOR_MANDI = "waiting_for_mandi"

@dataclass
class ConvState:
    stage:             Stage    = Stage.IDLE
    pending_crop:      str      = ""
    available_mandis:  list     = field(default_factory=list)
    pending_district:  str      = ""      # ← NEW: district being queried (may differ from home)
    farmer_state:      str      = ""
    farmer_district:   str      = ""
    farmer_lat:        float    = 0.0
    farmer_lon:        float    = 0.0
    field_name:        str      = ""
    city:              str      = ""
    thread_id:         str      = ""
    msg_count:         int      = 0
    summary:           str      = ""
    history:           list     = field(default_factory=list)
    last_msg_time:     object   = None    # ← NEW: datetime of last message

# ══════════════════════════════════════════════════════════
#  LLM
# ══════════════════════════════════════════════════════════
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    api_key=os.getenv("GROK_API_KEY"),
)

# ══════════════════════════════════════════════════════════
#  DB HELPERS
# ══════════════════════════════════════════════════════════
async def db_get_or_create_thread(farmer_id: str) -> dict:   # ← NEW: returns dict with updated_at
    conn = await asyncpg.connect(DB_URL)
    try:
        await conn.execute(
            """
            INSERT INTO chat_threads (thread_id, farmer_id, summary)
            VALUES ($1, $2::uuid, '')
            ON CONFLICT (thread_id) DO NOTHING
            """,
            farmer_id, farmer_id
        )
        row = await conn.fetchrow(
            "SELECT summary, updated_at FROM chat_threads WHERE thread_id = $1",
            farmer_id
        )
        return {
            "summary":    row["summary"] or "" if row else "",
            "updated_at": row["updated_at"] if row else None,   # ← NEW
        }
    finally:
        await conn.close()


async def db_save_message(thread_id: str, role: str, content: str):
    conn = await asyncpg.connect(DB_URL)
    try:
        await conn.execute(
            "INSERT INTO chat_messages (thread_id, role, content) VALUES ($1, $2, $3)",
            thread_id, role, content
        )
        await conn.execute(
            "UPDATE chat_threads SET updated_at = NOW() WHERE thread_id = $1",
            thread_id
        )
    finally:
        await conn.close()


async def db_load_recent_messages(thread_id: str, limit: int = 10) -> list:
    conn = await asyncpg.connect(DB_URL)
    try:
        rows = await conn.fetch(
            """
            SELECT role, content FROM chat_messages
            WHERE thread_id = $1
            ORDER BY created_at DESC LIMIT $2
            """,
            thread_id, limit
        )
        return [(r["role"], r["content"]) for r in reversed(rows)]
    finally:
        await conn.close()


async def db_save_summary(thread_id: str, summary: str):
    conn = await asyncpg.connect(DB_URL)
    try:
        await conn.execute(
            "UPDATE chat_threads SET summary = $2, updated_at = NOW() WHERE thread_id = $1",
            thread_id, summary
        )
    finally:
        await conn.close()

# ══════════════════════════════════════════════════════════
#  MEMORY
# ══════════════════════════════════════════════════════════
async def maybe_summarise(conv: ConvState):
    if conv.msg_count < MAX_MESSAGES:
        return

    print("\n📝 [Memory] Message limit reached — generating summary...")

    to_compress = conv.history[:SUMMARIZE_AFTER]
    to_keep     = conv.history[SUMMARIZE_AFTER:]
    convo_text  = "\n".join(f"{r.upper()}: {c}" for r, c in to_compress)

    # ← CHANGE 2: Enhanced prompt that explicitly stores prices + weather days
    prompt = f"""You are summarising a KisanSaathi agricultural assistant conversation.

Previous summary (if any): {conv.summary or "None"}

New messages to add to summary:
{convo_text}

Write a concise summary (max 200 words). You MUST include if they appeared:
1. PRICES: crop name, mandi name, district, exact price (₹/Quintal), date fetched
2. WEATHER: location, number of days forecasted, key highlights (rain mm, temp range, wind)
3. Districts/mandis the farmer asked about (including non-home districts)
4. Any farming decisions or advice given

Skip irrelevant small talk. Do NOT say "This is a summary"."""

    resp = await llm.ainvoke([HumanMessage(content=prompt)])
    conv.summary   = resp.content.strip()
    conv.history   = to_keep
    conv.msg_count = len(to_keep)

    await db_save_summary(conv.thread_id, conv.summary)
    print(f"📝 [Memory] Summary saved ({len(conv.summary)} chars). Keeping {len(to_keep)} recent messages.")


def add_to_history(conv: ConvState, role: str, content: str):
    conv.history.append((role, content))
    conv.msg_count += 1


# ← NEW: Change 3 helper
def is_stale_session(conv: ConvState) -> bool:
    """Returns True if last message was more than GAP_MINUTES ago."""
    if conv.last_msg_time is None:
        return False
    now  = datetime.now(timezone.utc)
    last = conv.last_msg_time
    if last.tzinfo is None:
        last = last.replace(tzinfo=timezone.utc)
    gap_min = (now - last).total_seconds() / 60
    if gap_min > GAP_MINUTES:
        print(f"\n⏰ [Session] Gap: {gap_min:.1f} min > {GAP_MINUTES} min → will use summary for context")
        return True
    return False

# ══════════════════════════════════════════════════════════
#  DATA FETCHERS
# ══════════════════════════════════════════════════════════
async def fetch_farmer_profile(farmer_id: str, conv: ConvState):
    conn = await asyncpg.connect(DB_URL)
    try:
        farmer = await conn.fetchrow(
            "SELECT state_name, dist_name FROM farmers WHERE id = $1", farmer_id
        )
        fld = await conn.fetchrow(
            """SELECT field_name, city_name, center_lat, center_lon
               FROM farm_fields WHERE farmer_id = $1 LIMIT 1""",
            farmer_id
        )
    finally:
        await conn.close()

    if farmer:
        conv.farmer_state    = farmer["state_name"]
        conv.farmer_district = farmer["dist_name"]
    if fld:
        conv.field_name = fld["field_name"]
        conv.city       = fld["city_name"]
        conv.farmer_lat = float(fld["center_lat"])
        conv.farmer_lon = float(fld["center_lon"])


def get_mandis_for_location(state: str, district: str) -> list:
    with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    for s_key, districts in data.items():
        if s_key.lower() == state.lower():
            for d_key, mandis in districts.items():
                if d_key.lower() == district.lower():
                    return mandis
    return []


async def fetch_mandi_price(state: str, mandi_name: str, crop: str) -> str:
    print(f"\n🌐 [API] Agmarknet → state={state} | mandi={mandi_name} | crop={crop}")
    clean  = mandi_name.replace("APMC", "").strip()
    params = {
        "api-key":            os.getenv("AGMARKNET_API_KEY"),
        "format":             "json",
        "filters[state]":     state,
        "filters[market]":    clean,
        "filters[commodity]": crop,
    }
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
    except httpx.TimeoutException:
        return "⚠️  Government server ne respond nahi kiya. Thodi der baad try karein."

    if resp.status_code != 200:
        return f"⚠️  API Error {resp.status_code}."

    records = resp.json().get("records", [])
    if not records:
        return (
            f"⚠️  Aaj {crop} ka data {clean} mandi mein available nahi hai.\n"
            "Kal try karein ya koi aur mandi chunein."
        )
    r = records[0]
    return (
        f"✅ Aaj Ka Bhav\n"
        f"━━━━━━━━━━━━━━━━━━━━━━\n"
        f"🌾 Fasal  : {r['commodity']}\n"
        f"🏪 Mandi  : {r['market']}\n"
        f"💰 Bhav   : ₹{r['modal_price']} / Quintal\n"
        f"📅 Tarikh : {r['arrival_date']}\n"
        f"━━━━━━━━━━━━━━━━━━━━━━"
    )


async def fetch_weather_data(lat: float, lon: float, days: int) -> dict:
    print(f"\n🌐 [API] Open-Meteo → lat={lat} | lon={lon} | days={days}")
    params = {
        "latitude":      lat,
        "longitude":     lon,
        "daily":         "temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max",
        "forecast_days": days,
        "timezone":      "Asia/Kolkata",
    }
    async with httpx.AsyncClient() as client:
        resp = await client.get(OPEN_METEO_URL, params=params, timeout=15.0)
    resp.raise_for_status()
    return resp.json()

# ══════════════════════════════════════════════════════════
#  LLM HELPERS
# ══════════════════════════════════════════════════════════
async def detect_intent(user_msg: str) -> dict:
    # ← CHANGE 1: added asked_district extraction
    prompt = f"""Analyze this farmer message and respond ONLY with JSON.

Message: "{user_msg}"

Rules:
- intent: one of ["price", "weather", "general"]
  • price   = asking about mandi price/bhav/rate/daam
  • weather = asking about mausam/barish/garmi/sardi/aandhi/temperature
  • general = anything else
- crop: crop name in English if mentioned, else ""
- days: number of days for weather (aaj=1, kal=2, teen din=3, hafte=7). 0 if not mentioned.
- asked_district: if farmer explicitly mentions a LOCATION like "Chandigarh", "Agra", "Lucknow"
  extract it in Title Case. This is WHERE they are asking ABOUT. Else "".

JSON only:
{{"intent": "price/weather/general", "crop": "", "days": 0, "asked_district": ""}}"""

    resp = await llm.ainvoke([HumanMessage(content=prompt)])
    try:
        text   = resp.content.strip().replace("```json","").replace("```","").strip()
        result = json.loads(text)
        days   = int(result.get("days", 0))
        if days <= 0 or days > 16:
            days = 3
        return {
            "intent":         result.get("intent", "general"),
            "crop":           result.get("crop", "").strip(),
            "days":           days,
            "asked_district": result.get("asked_district", "").strip(),   # ← NEW
        }
    except Exception:
        kw_price   = ["bhav","price","rate","daam","mandi","khareed","bech"]
        kw_weather = ["mausam","barish","baarish","weather","garmi","sardi","aandhi"]
        if any(k in user_msg.lower() for k in kw_price):
            return {"intent": "price",   "crop": "", "days": 3, "asked_district": ""}
        if any(k in user_msg.lower() for k in kw_weather):
            return {"intent": "weather", "crop": "", "days": 3, "asked_district": ""}
        return {"intent": "general", "crop": "", "days": 3, "asked_district": ""}


async def extract_crop(user_msg: str) -> str:
    prompt = f"""Farmer was asked which crop they want. Their reply: "{user_msg}"
Extract ONLY the English crop name (e.g. Wheat, Onion, Tomato).
If no crop found respond: NONE"""
    resp = await llm.ainvoke([HumanMessage(content=prompt)])
    crop = resp.content.strip()
    return "" if crop.upper() == "NONE" else crop


def fuzzy_match_mandi(user_input: str, mandis: list) -> str:
    user_input = user_input.strip()
    if user_input.isdigit():
        idx = int(user_input) - 1
        return mandis[idx] if 0 <= idx < len(mandis) else ""
    for m in mandis:
        if user_input.lower() in m.lower() or m.lower() in user_input.lower():
            return m
    return ""


async def llm_answer_weather(user_msg: str, weather_data: dict,
                              field_name: str, city: str, days: int,
                              summary: str) -> str:
    daily = weather_data.get("daily", {})
    dates = daily.get("time", [])
    t_max = daily.get("temperature_2m_max", [])
    t_min = daily.get("temperature_2m_min", [])
    rain  = daily.get("precipitation_sum", [])
    wind  = daily.get("windspeed_10m_max", [])

    data_lines = []
    for i in range(min(days, len(dates))):
        data_lines.append(
            f"  {dates[i]}: Temp {t_min[i]}–{t_max[i]}°C | Rain {rain[i]}mm | Wind {wind[i]} km/h"
        )

    context = f"Previous context: {summary}" if summary else ""

    prompt = f"""Tu KisanSaathi hai — helpful agricultural assistant for Indian farmers.

{context}
Farmer ka khet: {field_name}, {city}
Farmer ka sawaal: "{user_msg}"

LIVE weather data ({days} din ke liye) — abhi API se fetch hua:
{chr(10).join(data_lines)}

Hinglish mein jawab de (Hindi + English mix):
1. Farmer ke sawaal ka SEEDHA jawab pehle
2. Numbers simple bhasha mein explain kar
3. Khet ke liye relevant advice de
4. 4-6 lines max, warm desi tone
Bullet points mat use karna."""

    resp = await llm.ainvoke([HumanMessage(content=prompt)])
    return resp.content.strip()


# ← CHANGE 3: stale param added
async def llm_general_answer(user_msg: str, summary: str, history: list,
                              stale: bool = False) -> str:
    recent      = history[-4:] if len(history) > 4 else history
    recent_text = "\n".join(f"{r.upper()}: {c}" for r, c in recent)

    context = ""
    if summary:
        context += f"Conversation summary (pehle ki baatein):\n{summary}\n\n"
    if recent_text:
        context += f"Recent messages:\n{recent_text}"

    # ← NEW: stale session instruction
    stale_note = ""
    if stale:
        stale_note = (
            "\nNOTE: Farmer 10+ minute baad wapas aaya hai. "
            "Agar unka sawaal pehle ki baat se related lagta hai "
            "(jaise price ya mausam ki update pooch raha ho), "
            "toh summary se context lekar jawab do. "
            "Naya sawaal ho toh fresh answer do."
        )

    prompt = f"""Tu KisanSaathi hai — helpful agricultural assistant for Indian farmers.

{context}
{stale_note}

Farmer ka naya sawaal: "{user_msg}"

Hinglish mein (Hindi + English) helpful answer de. Short rakh (max 4-5 lines).
Summary se context lo but CURRENT sawaal ka jawab priority hai."""

    resp = await llm.ainvoke([HumanMessage(content=prompt)])
    return resp.content.strip()

# ══════════════════════════════════════════════════════════
#  MAIN MESSAGE HANDLER
# ══════════════════════════════════════════════════════════
async def handle_message(user_msg: str, conv: ConvState) -> str:

    # ← CHANGE 3: check stale at the top of every message
    stale = is_stale_session(conv)

    # ── STAGE: Waiting for crop ───────────────────────────
    if conv.stage == Stage.WAITING_FOR_CROP:
        crop = await extract_crop(user_msg)
        if not crop:
            return "Maafi chahta hun, samajh nahi aaya. Kripya fasal ka naam batayein — jaise Wheat, Onion, Tomato."

        conv.pending_crop = crop
        # ← CHANGE 1: use pending_district if cross-district query was made
        query_district    = conv.pending_district or conv.farmer_district
        mandis            = get_mandis_for_location(conv.farmer_state, query_district)

        if not mandis:
            conv.stage            = Stage.IDLE
            conv.pending_district = ""
            return f"{query_district} district ke liye mandi data abhi available nahi hai."

        conv.available_mandis = mandis
        conv.stage            = Stage.WAITING_FOR_MANDI
        mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(mandis))
        return (
            f"📍 {query_district} ({conv.farmer_state}) ki APMCs — {crop}:\n\n"
            f"{mandi_list}\n\n"
            "Kaunsi mandi ka bhav chahiye? (Number ya naam type karein)"
        )

    # ── STAGE: Waiting for mandi selection ───────────────
    if conv.stage == Stage.WAITING_FOR_MANDI:
        matched = fuzzy_match_mandi(user_msg, conv.available_mandis)
        if not matched:
            mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(conv.available_mandis))
            return f"Yeh mandi nahi mili. List mein se number ya naam chunein:\n\n{mandi_list}"

        result = await fetch_mandi_price(conv.farmer_state, matched, conv.pending_crop)

        conv.stage            = Stage.IDLE
        conv.pending_crop     = ""
        conv.pending_district = ""     # ← NEW: reset after use
        conv.available_mandis = []
        return result

    # ── STAGE: IDLE ───────────────────────────────────────
    intent_data     = await detect_intent(user_msg)
    intent          = intent_data["intent"]
    asked_district  = intent_data.get("asked_district", "")   # ← NEW

    # ── WEATHER ──────────────────────────────────────────
    if intent == "weather":
        days = intent_data["days"]

        # ← CHANGE 3: stale + weather → answer from summary if possible
        if stale and conv.summary and "weather" in conv.summary.lower():
            return await llm_general_answer(user_msg, conv.summary, conv.history, stale=True)

        if not conv.farmer_lat:
            try:
                await fetch_farmer_profile(FARMER_ID, conv)
            except Exception as e:
                return f"⚠️ Database error: {e}"

        if not conv.farmer_lat:
            return "⚠️ Aapka khet registered nahi hai. Pehle khet register karein."

        try:
            weather_data = await fetch_weather_data(conv.farmer_lat, conv.farmer_lon, days)
        except Exception as e:
            return f"⚠️ Mausam server se data nahi aaya: {e}"

        return await llm_answer_weather(
            user_msg, weather_data, conv.field_name, conv.city, days, conv.summary
        )

    # ── MANDI PRICE ───────────────────────────────────────
    if intent == "price":
        crop = intent_data["crop"]

        if not conv.farmer_state:
            try:
                await fetch_farmer_profile(FARMER_ID, conv)
            except Exception as e:
                return f"⚠️ Database error: {e}"

        # ← CHANGE 1: Cross-district logic
        # If farmer asked about a different district → use it, keep same state
        if asked_district and asked_district.lower() != conv.farmer_district.lower():
            query_district = asked_district
            print(f"\n📍 [Cross-district] Farmer asked about '{asked_district}' (home: '{conv.farmer_district}')")
        else:
            query_district = conv.farmer_district

        conv.pending_district = query_district   # ← save so crop stage can use it

        if not crop:
            conv.stage = Stage.WAITING_FOR_CROP
            if asked_district:
                return f"Aap {asked_district} mein kis fasal (crop) ka bhav jaanna chahte hain? (e.g. Wheat, Onion, Tomato)"
            return "Aap kis fasal (crop) ka bhav jaanna chahte hain? (e.g. Wheat, Onion, Tomato)"

        conv.pending_crop = crop
        mandis = get_mandis_for_location(conv.farmer_state, query_district)

        if not mandis:
            return f"{query_district} district ke liye mandi data abhi available nahi hai."

        conv.available_mandis = mandis
        conv.stage            = Stage.WAITING_FOR_MANDI
        mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(mandis))
        return (
            f"📍 {query_district} ({conv.farmer_state}) ki APMCs — {crop}:\n\n"
            f"{mandi_list}\n\n"
            "Kaunsi mandi ka bhav chahiye? (Number ya naam type karein)"
        )

    # ── GENERAL ───────────────────────────────────────────
    return await llm_general_answer(user_msg, conv.summary, conv.history, stale=stale)

# ══════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════
async def chat_with_farmer():
    print(f"\n🌾 KisanSaathi | Farmer: {FARMER_ID}")
    print("━" * 50)

    conv           = ConvState()
    conv.thread_id = FARMER_ID

    # ← CHANGE 3: db_get_or_create_thread now returns dict with updated_at
    thread_data        = await db_get_or_create_thread(FARMER_ID)
    conv.summary       = thread_data["summary"]
    conv.last_msg_time = thread_data["updated_at"]   # ← NEW: last chat time from DB

    recent_msgs    = await db_load_recent_messages(FARMER_ID, limit=KEEP_RECENT)
    conv.history   = recent_msgs
    conv.msg_count = len(recent_msgs)

    if conv.summary:
        print(f"📖 [Resuming chat] Summary loaded ({len(conv.summary)} chars)")
    if recent_msgs:
        print(f"📖 [Resuming chat] {len(recent_msgs)} recent messages loaded")
        print("\n--- Previous Messages ---")
        for role, content in recent_msgs:
            prefix = "👨‍🌾 Farmer" if role == "human" else "🤖 KisanSaathi"
            print(f"{prefix}: {content}")
        print("--- End of Previous Messages ---\n")

    # ← NEW: show gap info at startup
    if conv.last_msg_time:
        last = conv.last_msg_time
        if last.tzinfo is None:
            last = last.replace(tzinfo=timezone.utc)
        gap_min = (datetime.now(timezone.utc) - last).total_seconds() / 60
        if gap_min > GAP_MINUTES:
            print(f"⏰ [Session] Last chat was {gap_min:.0f} min ago — context will be used from summary.\n")

    print("\n🤖 KisanSaathi: Namaste! Main KisanSaathi hun. Aaj kya jaanna chahte hain?\n")

    while True:
        try:
            user_input = input("👨‍🌾 Farmer: ").strip()
        except EOFError:
            break

        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "band karo", "bye"):
            print("🤖 KisanSaathi: Dhanyavad! Jai Jawan Jai Kisan 🌾")
            break

        print("🤖 KisanSaathi: ", end="", flush=True)
        response = await handle_message(user_input, conv)

        for word in response.split(" "):
            print(word, end=" ", flush=True)
            await asyncio.sleep(0.03)
        print()

        # ← CHANGE 3: update last_msg_time after every message
        conv.last_msg_time = datetime.now(timezone.utc)

        add_to_history(conv, "human", user_input)
        add_to_history(conv, "ai",    response)
        await db_save_message(conv.thread_id, "human", user_input)
        await db_save_message(conv.thread_id, "ai",    response)

        await maybe_summarise(conv)
        print()


# Run in Jupyter:
await chat_with_farmer()

# Run as .py:
# if __name__ == "__main__":
#     asyncio.run(chat_with_farmer())



🌾 KisanSaathi | Farmer: c59f6f44-1a98-4eaa-8cf0-3581316a32bb
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📖 [Resuming chat] Summary loaded (623 chars)
📖 [Resuming chat] 2 recent messages loaded

--- Previous Messages ---
👨‍🌾 Farmer: 6
🤖 KisanSaathi: ⚠️  Aaj Wheat ka data Jarar mandi mein available nahi hai.
Kal try karein ya koi aur mandi chunein.
--- End of Previous Messages ---

⏰ [Session] Last chat was 19 min ago — context will be used from summary.


🤖 KisanSaathi: Namaste! Main KisanSaathi hun. Aaj kya jaanna chahte hain?

🤖 KisanSaathi: 
⏰ [Session] Gap: 19.5 min > 10 min → will use summary for context
Namaste, kaise ho? Aapki wheat ki kheti kaisi chal rahi hai? 

🤖 KisanSaathi: 📍 agra (Uttar Pradesh) ki APMCs — wheat:

  1. Achnera APMC
  2. Agra APMC
  3. Fatehabad APMC
  4. Fatehpur Sikri APMC
  5. Jagnair APMC
  6. Jarar APMC

Kaunsi mandi ka bhav chahiye? (Number ya naam type karein) 

🤖 KisanSaathi: 
🌐 [API] Agmarknet → state=Uttar Pradesh | mandi=Achnera APMC | cro